In [ ]:
import requests
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import nltk
from newspaper import Article
import re  # Untuk pembersihan teks
import numpy as np  # Untuk menangani array
from langdetect import detect, LangDetectException  # Untuk deteksi bahasa

nltk.download('punkt')

def google_search_bs(query, num_pages=10):
    hasil_pencarian = []
    base_url = "https://www.google.com/search"

    for page in range(num_pages):
        start = page * 50 
        params = {
            'q': query,
            'start': start,
            'hl': 'id',  
            'tbs': 'qdr:y' 
        }
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, seperti Gecko) Chrome/85.0.4183.121 Safari/537.36"
        }

        response = requests.get(base_url, headers=headers, params=params)
        soup = BeautifulSoup(response.text, 'html.parser')

        for g in soup.find_all('div', class_='g'):
            anchors = g.find_all('a')
            if anchors:
                link = anchors[0]['href']
                title = g.find('h3').text if g.find('h3') else 'Tidak ada judul'
                snippet = g.find('span', class_='aCOpRe').text if g.find('span', class_='aCOpRe') else 'Tidak ada cuplikan'
                hasil_pencarian.append({
                    'judul': title,
                    'link': link,
                    'cuplikan': snippet
                })
    return hasil_pencarian

def clean_text(text):
    """Membersihkan teks dengan menghapus URL, angka, dan karakter khusus."""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\d+', '', text)  
    text = re.sub(r'\W+', ' ', text) 
    return text.strip()

def fetch_and_summarize(url):
    try:
        artikel = Article(url)
        artikel.download()
        artikel.parse()
        artikel.nlp()
        return artikel.summary
    except Exception as e:
        print(f"Kesalahan saat mengambil ringkasan untuk {url}: {e}")
        return None

def kmeans_clustering(summaries, n_clusters=3):  
    vectorizer = TfidfVectorizer(ngram_range=(1, 2)) 
    X = vectorizer.fit_transform(summaries)

    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    labels = kmeans.fit_predict(X)

    return labels, X, vectorizer, kmeans

def get_cluster_keywords(kmeans, vectorizer, n_words=3):
    """Mengambil kata kunci dari setiap cluster berdasarkan bobot fitur."""
    keywords = {}
    for i in range(kmeans.n_clusters):
        center = kmeans.cluster_centers_[i]
        top_keywords_idx = center.argsort()[-n_words:][::-1]
        top_keywords = [vectorizer.get_feature_names_out()[idx] for idx in top_keywords_idx]
        keywords[i] = top_keywords
    return keywords

def is_indonesian(text):
    """Mendeteksi apakah teks dalam bahasa Indonesia."""
    try:
        return detect(text) == 'id'
    except LangDetectException:
        return False

def search_and_cluster(query, n_clusters=3):  
    hasil = google_search_bs(query)
    ringkasan = []

    dict_cluster = {i: [] for i in range(n_clusters)}  

    for result in hasil:
        summary = fetch_and_summarize(result['link'])
        if summary and is_indonesian(summary):  
            cleaned_summary = clean_text(summary)
            ringkasan.append(cleaned_summary)

    if not ringkasan:
        print("Tidak ada ringkasan yang tersedia untuk clustering.")
        return

    cluster_labels, X, vectorizer, kmeans = kmeans_clustering(ringkasan, n_clusters)

    for idx, result in enumerate(hasil):
        if idx < len(ringkasan): 
            dict_cluster[cluster_labels[idx]].append({
                'Judul': result['judul'],
                'URL': result['link'],
                'Cuplikan': result['cuplikan'],
                'Ringkasan': ringkasan[idx]
            })

    hasil_per_cluster = {nomor_cluster: items for nomor_cluster, items in dict_cluster.items() if items}

    cluster_keywords = get_cluster_keywords(kmeans, vectorizer)

    for nomor_cluster, items in hasil_per_cluster.items():
        print(f"Cluster {nomor_cluster}: {len(items)} item")

    return hasil_per_cluster, cluster_keywords

kluster, kata_kunci = search_and_cluster("Shin Tae-yong 2024")

for nomor_cluster, items in kluster.items():
    print(f"Cluster {nomor_cluster} (Kata Kunci: {kata_kunci[nomor_cluster]}):")
    for item in items:
        print(f"Judul: {item['Judul']}")
        print(f"URL: {item['URL']}")
        print(f"Cuplikan: {item['Cuplikan']}")
        print(f"Ringkasan: {item['Ringkasan']}")
        print("-" * 50)
